In [1081]:
                                                                        1) User-Based RecSys

In [ ]:
import pandas as pd
import numpy as np
 
# модуль sparse библиотеки scipy понадобится 
# для работы с разреженными матрицами (об этом ниже)
from scipy.sparse import csr_matrix
 
# из sklearn импортируем алгоритм k-ближайших соседей
from sklearn.neighbors import NearestNeighbors

In [1083]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

In [1084]:
# посмотрим на содержимое файла movies.csv
# дополнительно удалим столбец genres, он нам не нужен
# (параметр axis = 1 говорит, что мы работаем со столбцами, inplace = True, что изменения нужно сохранить)
movies.drop(['genres'], axis = 1, inplace = True)
movies.head(3)

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)


In [1085]:
# и ratings.csv (здесь также удаляем ненужный столбец timestamp)
ratings.drop(['timestamp'], axis = 1, inplace = True)
ratings.head(3)

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0


In [1088]:
user_item_matrix = ratings.pivot(index = 'movieId', columns = 'userId', values = 'rating')
user_item_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,NaN,4.0,NaN,4.5,NaN,NaN,NaN,...,4.0,NaN,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [1090]:
# параметр inplace = True опять же поможет сохранить результат
user_item_matrix.fillna(0, inplace = True)
user_item_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1091]:
# вначале сгруппируем (объединим) пользователей, возьмем только столбец rating 
# и посчитаем, сколько было оценок у каждого пользователя
users_votes = ratings.groupby('userId')['rating'].agg('count')
users_votes

userId
1       232
2        29
3        39
4       216
5        44
       ... 
606    1115
607     187
608     831
609      37
610    1302
Name: rating, Length: 610, dtype: int64

In [1092]:
users_votes.min()

20

In [1094]:
# сделаем то же самое, только для фильма
movies_votes = ratings.groupby('movieId')['rating'].agg('count')
movies_votes

movieId
1         215
2         110
3          52
4           7
5          49
         ... 
193581      1
193583      1
193585      1
193587      1
193609      1
Name: rating, Length: 9724, dtype: int64

In [1095]:
movies_votes.min()

1

In [1096]:
# теперь создадим фильтр (mask)
user_mask = users_votes[users_votes > 50].index
movie_mask = movies_votes[movies_votes > 10].index

In [1097]:
user_mask

Index([  1,   4,   6,   7,  10,  11,  15,  16,  17,  18,
       ...
       600, 601, 602, 603, 604, 605, 606, 607, 608, 610],
      dtype='int64', name='userId', length=378)

In [1098]:
movie_mask

Index([     1,      2,      3,      5,      6,      7,      9,     10,     11,
           12,
       ...
       159093, 164179, 166528, 168250, 168252, 174055, 176371, 177765, 179819,
       187593],
      dtype='int64', name='movieId', length=2121)

In [1100]:
# применим фильтры и отберем фильмы с достаточным количеством оценок,
user_item_matrix = user_item_matrix.loc[movie_mask,:]
 
# а также активных пользователей
user_item_matrix = user_item_matrix.loc[:,user_mask]

In [1102]:
user_item_matrix.head()

C:\Users\Nikita\anaconda3\Lib\site-packages\IPython\core\displayhook.py:281: UserWarning: Output cache limit (currently 1000 entries) hit.
Flushing oldest 200 entries.
  warn('Output cache limit (currently {sz} entries) hit.\n'


userId,1,4,6,7,10,11,15,16,17,18,...,600,601,602,603,604,605,606,607,608,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,4.5,0.0,0.0,2.5,0.0,4.5,3.5,...,2.5,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,5.0
2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,4.0,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0
3,4.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
5,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.5,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
6,4.0,0.0,4.0,0.0,0.0,5.0,0.0,0.0,0.0,4.0,...,0.0,0.0,3.0,4.0,3.0,0.0,0.0,0.0,0.0,5.0


In [1104]:
user_item_matrix.shape

(2121, 378)

In [1113]:
csr_data = csr_matrix(user_item_matrix.values)

In [1115]:
print(csr_data[:3,:5])

  (0, 0)	4.0
  (0, 3)	4.5
  (1, 2)	4.0
  (2, 0)	4.0
  (2, 2)	5.0


In [1117]:
user_item_matrix = user_item_matrix.rename_axis(None, axis = 1).reset_index()
user_item_matrix.head()

,movieId,1,4,6,7,10,11,15,16,17,...,600,601,602,603,604,605,606,607,608,610
0,1,4.0,0.0,0.0,4.5,0.0,0.0,2.5,0.0,4.5,...,2.5,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,5.0
1,2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0
2,3,4.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
3,5,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.5,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
4,6,4.0,0.0,4.0,0.0,0.0,5.0,0.0,0.0,0.0,...,0.0,0.0,3.0,4.0,3.0,0.0,0.0,0.0,0.0,5.0


In [1119]:
# создадим объект класса NearestNeighbors
knn = NearestNeighbors(metric = 'cosine',
                       algorithm = 'brute',
                       n_neighbors = 20,
                       n_jobs = -1)
 
# обучим модель
knn.fit(csr_data)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=20)

In [1121]:
recommendations = 10
search_word = 'Matrix'

In [1123]:
# для начала найдем фильм в заголовках датафрейма movies
movie_search = movies[movies['title'].str.contains(search_word)]
movie_search

,movieId,title
1939,2571,"Matrix, The (1999)"
4351,6365,"Matrix Reloaded, The (2003)"
4639,6934,"Matrix Revolutions, The (2003)"


In [1125]:
# вариантов может быть несколько, для простоты всегда будем брать первый вариант
# через iloc[0] мы берем первую строку столбца ['movieId']
movie_id = movie_search.iloc[0]['movieId']

In [1127]:
movie_id

2571

In [1131]:
# далее по индексу фильма в датасете movies найдем соответствующий индекс
# в матрице предпочтений, так как там мы сбросили индексы
movie_id = user_item_matrix[user_item_matrix['movieId'] == movie_id].index[0]
movie_id

IndexError: index 0 is out of bounds for axis 0 with size 0

In [813]:
print(csr_data[movie_id])

  (0, 0)	5.0
  (0, 1)	1.0
  (0, 4)	0.5
  (0, 6)	4.0
  (0, 7)	3.5
  (0, 8)	5.0
  (0, 9)	4.5
  (0, 10)	4.0
  (0, 12)	4.0
  (0, 15)	4.0
  (0, 17)	4.0
  (0, 20)	5.0
  (0, 21)	3.0
  (0, 24)	5.0
  (0, 26)	2.0
  (0, 27)	5.0
  (0, 29)	5.0
  (0, 31)	2.5
  (0, 33)	5.0
  (0, 34)	5.0
  (0, 36)	1.0
  (0, 37)	5.0
  (0, 38)	3.5
  (0, 39)	4.0
  (0, 40)	5.0
  :	:
  (0, 335)	5.0
  (0, 336)	5.0
  (0, 339)	4.5
  (0, 340)	4.0
  (0, 341)	5.0
  (0, 346)	4.5
  (0, 349)	5.0
  (0, 351)	5.0
  (0, 352)	5.0
  (0, 353)	5.0
  (0, 357)	5.0
  (0, 358)	4.0
  (0, 360)	4.0
  (0, 361)	5.0
  (0, 363)	2.5
  (0, 364)	5.0
  (0, 365)	4.0
  (0, 367)	5.0
  (0, 368)	3.0
  (0, 369)	5.0
  (0, 371)	5.0
  (0, 374)	5.0
  (0, 375)	5.0
  (0, 376)	5.0
  (0, 377)	5.0


In [221]:
distances, indices = knn.kneighbors(csr_data[movie_id], n_neighbors = recommendations + 1)

In [233]:
# уберем лишние измерения через squeeze() и преобразуем массивы в списки с помощью tolist()
indices_list = indices.squeeze().tolist()
distances_list = distances.squeeze().tolist()

In [235]:
indices_list

[901, 1002, 442, 454, 124, 735, 954, 1362, 1157, 1536, 978]

In [237]:
distances_list

[0.0,
 0.22982440568634488,
 0.25401128310081567,
 0.27565616686043737,
 0.2776088577731709,
 0.2869100842838125,
 0.2911101181714415,
 0.31393358217709477,
 0.31405925934381695,
 0.3154800434449465,
 0.31748544046311844]

In [239]:
# далее с помощью функций zip() и list() преобразуем списки
indices_distances = list(zip(indices_list, distances_list))

In [253]:
# в набор кортежей (tuple)
print(type(indices_distances[0]))
 
# и посмотрим на первые три пары/кортежа
print(indices_distances[:3])

<class 'tuple'>
[(901, 0.0), (1002, 0.22982440568634488), (442, 0.25401128310081567)]


In [255]:
# остается отсортировать список по расстояниям через key = lambda x: x[1] (то есть по второму элементу)
# в возрастающем порядке reverse = False
indices_distances_sorted = sorted(indices_distances, key = lambda x: x[1], reverse = False)
 
# и убрать первый элемент с индексом 901 (потому что это и есть "Матрица")
indices_distances_sorted = indices_distances_sorted[1:]
indices_distances_sorted

[(1002, 0.22982440568634488),
 (442, 0.25401128310081567),
 (454, 0.27565616686043737),
 (124, 0.2776088577731709),
 (735, 0.2869100842838125),
 (954, 0.2911101181714415),
 (1362, 0.31393358217709477),
 (1157, 0.31405925934381695),
 (1536, 0.3154800434449465),
 (978, 0.31748544046311844)]

In [257]:
# создадим пустой список, в который будем помещать название фильма и расстояние до него
recom_list = []
 
# в цикле будем поочередно проходить по кортежам
for ind_dist in indices_distances_sorted:
 
    # искать movieId в матрице предпочтений
    matrix_movie_id = user_item_matrix.iloc[ind_dist[0]]['movieId']
 
    # выяснять индекс этого фильма в датафрейме movies
    id = movies[movies['movieId'] == matrix_movie_id].index
 
    # брать название фильма и расстояние до него
    title = movies.iloc[id]['title'].values[0]
    dist = ind_dist[1]
 
    # помещать каждую пару в питоновский словарь,
    # который, в свою очередь, станет элементом списка recom_list
    recom_list.append({'Title' : title, 'Distance' : dist})

In [259]:
recom_list[0]

{'Title': 'Fight Club (1999)', 'Distance': 0.22982440568634488}

In [261]:
recom_df = pd.DataFrame(recom_list, index = range(1, recommendations + 1))
recom_df

,Title,Distance
1,Fight Club (1999),0.229824
2,Star Wars: Episode V - The Empire Strikes Back...,0.254011
3,Star Wars: Episode VI - Return of the Jedi (1983),0.275656
4,Star Wars: Episode IV - A New Hope (1977),0.277609
5,Saving Private Ryan (1998),0.286910
6,"Sixth Sense, The (1999)",0.291110
7,"Lord of the Rings: The Fellowship of the Ring,...",0.313934
8,Gladiator (2000),0.314059
9,"Lord of the Rings: The Return of the King, The...",0.315480
10,American Beauty (1999),0.317485


In [ ]:
                                                        2) item-based RecSys для конкретного пользователя

In [1199]:
#Этап загрузки данных и оубчения модели
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Подгружаем данные и убираем лишние столбцы
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
movies.drop(['genres'], axis=1, inplace=True)
ratings.drop(['timestamp'], axis=1, inplace=True)

# Создаем матрицу предпочтений (movieId как строки, userId как столбцы)
user_item_matrix = ratings.pivot_table(index='movieId', columns='userId', values='rating', fill_value=0)

# Фильтруем данные (активные пользователи и фильмы)
users_votes = ratings.groupby('userId')['rating'].agg('count')
movies_votes = ratings.groupby('movieId')['rating'].agg('count')
user_mask = users_votes[users_votes > 50].index
movie_mask = movies_votes[movies_votes > 10].index
user_item_matrix = user_item_matrix.loc[movie_mask, user_mask]

# Преобразуем матрицу в разреженный формат (транспонируем)
csr_data = csr_matrix(user_item_matrix.T)

# Строим и обучаем модель
knn = NearestNeighbors(metric='cosine',
                       algorithm='brute',
                       n_neighbors=20,
                       n_jobs=-1)
knn.fit(csr_data)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=20)

In [1200]:
# Ввод пользователя для которого нужно сделать рекомендацию
user_id_for_recommendation = 1 # Укажите id нужного пользователя
# Получаем список оцененных фильмов пользователем
user_ratings = user_item_matrix[user_id_for_recommendation].values.T

In [1203]:
user_ratings

array([4., 0., 4., ..., 0., 0., 0.])

In [1281]:
# Получаем индексы фильмов, которые пользователь оценил
rated_movie_indices = np.where(user_ratings != 0)[0]

if len(rated_movie_indices) == 0:
    print(f"Пользователь {user_id_for_recommendation} не оставил оценок. Нет рекомендаций.")
else:
    # Получаем индексы Id фильмов, которые оценил пользователь (Pandas Index)
    rated_movie_ids = user_item_matrix.index[rated_movie_indices]
    
    # Вычисляем номер строки(ID пользователя) в user_item_matrix (np.ndArray)
    rated_movie_indices_in_csr = np.array([user_item_matrix.columns.get_loc(user_id_for_recommendation)])

    # Вычисляем средний вектор пользователя и преобразуем его в массив NumPy
    user_vector = np.asarray(csr_data[rated_movie_indices_in_csr].mean(axis=0))

In [1285]:
rated_movie_ids

Index([   1,    3,    6,   47,   50,   70,  101,  110,  151,  157,
       ...
       3617, 3639, 3671, 3702, 3703, 3740, 3744, 3793, 3809, 5060],
      dtype='int64', name='movieId', length=209)

In [1277]:
print(rated_movie_indices_in_csr)

[0]


In [1279]:
print(user_vector)

[[0. 0. 0. ... 0. 0. 0.]]


In [1253]:
print(user_vector.shape)

(1, 2121)


In [1255]:
csr_data[rated_movie_indices_in_csr]

<1x2121 sparse matrix of type '<class 'numpy.float64'>'
	with 209 stored elements in Compressed Sparse Row format>

In [1257]:
print(csr_data[rated_movie_indices_in_csr])

  (0, 0)	4.0
  (0, 2)	4.0
  (0, 4)	4.0
  (0, 34)	5.0
  (0, 36)	5.0
  (0, 43)	3.0
  (0, 52)	5.0
  (0, 56)	4.0
  (0, 68)	5.0
  (0, 71)	5.0
  (0, 77)	5.0
  (0, 100)	5.0
  (0, 103)	3.0
  (0, 108)	5.0
  (0, 112)	4.0
  (0, 124)	5.0
  (0, 141)	3.0
  (0, 149)	3.0
  (0, 156)	5.0
  (0, 164)	4.0
  (0, 169)	4.0
  (0, 172)	5.0
  (0, 175)	4.0
  (0, 205)	4.0
  (0, 210)	5.0
  :	:
  (0, 1060)	1.0
  (0, 1069)	3.0
  (0, 1072)	3.0
  (0, 1075)	5.0
  (0, 1092)	5.0
  (0, 1110)	5.0
  (0, 1123)	4.0
  (0, 1124)	4.0
  (0, 1125)	5.0
  (0, 1126)	5.0
  (0, 1127)	5.0
  (0, 1136)	4.0
  (0, 1139)	4.0
  (0, 1147)	4.0
  (0, 1157)	5.0
  (0, 1162)	4.0
  (0, 1169)	4.0
  (0, 1173)	5.0
  (0, 1182)	5.0
  (0, 1183)	5.0
  (0, 1188)	4.0
  (0, 1190)	4.0
  (0, 1201)	5.0
  (0, 1203)	4.0
  (0, 1371)	5.0


In [1259]:
# Строим рекомендации для усредненного вектора пользователя
recommendations = 10
distances, indices = knn.kneighbors(user_vector, n_neighbors=recommendations + 1)
indices_list = indices.squeeze().tolist()
distances_list = distances.squeeze().tolist()

indices_distances = list(zip(indices_list, distances_list))
indices_distances_sorted = sorted(indices_distances, key=lambda x: x[1], reverse=False)
indices_distances_sorted = indices_distances_sorted[1:]

In [1261]:
indices_distances_sorted

[(229, 0.6047172774114625),
 (193, 0.609453847277913),
 (57, 0.6140971386896643),
 (165, 0.6182937397576205),
 (177, 0.6192237602240835),
 (34, 0.6209022250178893),
 (133, 0.6224025957331519),
 (10, 0.6255052165585675),
 (290, 0.628291406996312),
 (367, 0.6290824151242224)]

In [1263]:
recom_list = []
for ind_dist in indices_distances_sorted:
    matrix_movie_id = user_item_matrix.index[ind_dist[0]]
    id = movies[movies['movieId'] == matrix_movie_id].index
    title = movies.iloc[id]['title'].values[0]
    dist = ind_dist[1]
    recom_list.append({'Title': title, 'Distance': dist})

recom_df = pd.DataFrame(recom_list, index=range(1, recommendations + 1))
print(f"Рекомендации для пользователя {user_id_for_recommendation}:")
print(recom_df)

Рекомендации для пользователя 1:
                                        Title  Distance
1                     Perfect World, A (1993)  0.604717
2                     Another Stakeout (1993)  0.609454
3                          Taxi Driver (1976)  0.614097
4                          Client, The (1994)  0.618294
5   Naked Gun 33 1/3: The Final Insult (1994)  0.619224
6                 Seven (a.k.a. Se7en) (1995)  0.620902
7               Miracle on 34th Street (1994)  0.622403
8                                Nixon (1995)  0.625505
9                     Oliver & Company (1988)  0.628291
10                  African Queen, The (1951)  0.629082
